In [23]:
import json
import math
# 加载词典
# 加载两个词的共现的词频信息
word_distri = json.load(open('work/word_distri.json', 'r', encoding='utf-8'))
word_set = json.load(open('work/wordSet.json', 'r', encoding='utf-8'))
# sentences中为待分词句子
sentences = ["武汉市长江大桥","热烈欢迎新老师生前来就餐", "近年来自动化研究所取得的成就斐然","门把手坏了","用正向最大匹配法和逆向最大匹配法进行中文分词的结果是完全一样且正确的"]
# T_sentences 中为sentences中各句话的标准分词结果
T_sentences=["武汉市/长江大桥","热烈/欢迎/新/老/师生/前来/就餐", "近年来/自动化研究所/取得/的/成就斐然","门/把手/坏/了","用/正向最大匹配法/和/逆向最大匹配法/进行/中文/分词/的/结果/是/完全/一样/且/正确/的"]

**原句：工作的和尚未工作的都应该好好考虑一下人生大事**

**标准分词 A：['工作',' 的',' 和',' 尚未','工作','的',' 都',' 应该',' 好好',' 考虑',' 一下',' 人生',' 大事']**

**标准区间 A：[1,2],[3,3],[4,4],[5,6],[7,8],[9,9],[10,10],[11,12],[13,14],[15,16],[17,18],[19,20],[21,22]**

**分词结果 B：['工作',' 的','和尚','未工作 ','的 ','都',' 应该',' 好好考虑',' 一下',' 人生大事']**

**分词区间 B：[1,2],[3,3],[4,5],[6,7,8],[9,9],[10,10],[11,12],[13,14,15,16],[17,18],[19,20,21,22]**

**重复词语 A∩B：['工作',' 的',' 的',' 都',' 应该',' 一下']**

**重复区间 A∩B：[1,2], [3,3], [9,9],[10,10],[11,12],[17,18]**

**词典：['工作', '尚未', '的', '和', '青年', '都', '应该', '好好考虑', '自己', '人生', '大事']**

**分词评价指标：**
![](https://ai-studio-static-online.cdn.bcebos.com/dd1e8e316df44215b729ddacb553927508e2fd52c160490680e3ec9cef37bb7c)

![](https://ai-studio-static-online.cdn.bcebos.com/789d89d998d04ff192bddde4f0e20b0a306ee076d02740369f41c2b396e540e2)

![](https://ai-studio-static-online.cdn.bcebos.com/00617f09a02c4c48a9109d36124486ef9ade5e4fcf40491ab8f07d5debad5a02)



**名称解释：OOV（Out Of Vocabulary）指的是“未登录词”的简称，即词典中不存在的词。出现OOV的原因一方面可能确实是因为产生了有意义的新词而词典并没有收录；另一方面可能就是因为分词器产生了错误无意义的分词结果。**

**名称解释：IV（In Vocabulary）指的是“登陆词”，也就是已经存在于字典中的词。**

**说明： 前面三种指标可以用来衡量任意一种分词器的分词效果；而后两种指标则是用来衡量基于词典分词模型的分词效果。OOV recall越低也就意味着词典分词器对有意义新词的发现或者说查找能力越低。IV 召回率用来衡量词典中的词被正确找回的概率。如果IV召回率低，就说明字典分词器对词典中的词汇都无法百分之百的发现或者找回，说明其消歧能力不好。**

**例句分词结果评价指标如下：**
![](https://ai-studio-static-online.cdn.bcebos.com/c8fbef2d58824fa0b4e86a8613539e24ca5e784a47d744f7aea3ca73ad09ef31)




**请编写函数计算分词精确率**

In [24]:
# 以字符区间为单位评估分词结果，避免重复词语影响统计。
def _spans(words):
    spans, start = set(), 0
    for word in words:
        end = start + len(word)
        spans.add((start, end))
        start = end
    return spans

def precision(sentence, gold, pred):
    g, p = _spans(gold), _spans(pred)
    return len(g & p) / len(p) if p else 0.0

print('示例精确率函数已定义')

示例精确率函数已定义


**请编写函数计算分词召回率**

In [25]:
def recall(sentence, gold, pred):
    g, p = _spans(gold), _spans(pred)
    return len(g & p) / len(g) if g else 0.0

print('示例召回率：', recall('', ['ab'], ['ab']))

示例召回率： 1.0


**请编写函数计算分词F1值**

In [26]:
def f1_score(sentence, gold, pred):
    p, r = precision(sentence, gold, pred), recall(sentence, gold, pred)
    return 2 * p * r / (p + r) if p + r else 0.0

print('示例F1：', f1_score('', ['ab'], ['ab']))

示例F1： 1.0


**请编写函数计算OOV召回率**

In [27]:
def oov_recall(gold, pred, dictionary):
    # OOV 只统计标准结果中不在词典的词。
    target = {w for w in gold if w not in dictionary}
    return len(target & set(pred)) / len(target) if target else 0.0

print('OOV召回率函数已定义')

OOV召回率函数已定义


**请编写函数计算IV召回率**

In [28]:
def iv_recall(gold, pred, dictionary):
    target = {w for w in gold if w in dictionary}
    return len(target & set(pred)) / len(target) if target else 0.0

print('IV召回率函数已定义')

IV召回率函数已定义


**请用结巴分词的精确模式对sentences 中的话进行分词，并输出每句话分词结果的精确率、召回率、F1值**

In [29]:
def show_metrics(name, pred_words, dictionary=None):
    dictionary = set(dictionary or [])

    print('\n' + name)

    for i, (sentence, target) in enumerate(zip(sentences, T_sentences)):
        gold = target.split('/')
        pred = pred_words[i]

        p = precision(sentence, gold, pred)
        r = recall(sentence, gold, pred)
        f1 = f1_score(sentence, gold, pred)
        oov = oov_recall(gold, pred, dictionary) if dictionary else 0.0
        iv = iv_recall(gold, pred, dictionary) if dictionary else 0.0

        print(f'\n第 {i + 1} 句：{sentence}')
        print('标准：', gold)
        print('预测：', pred)
        print(
            'P={:.4f} R={:.4f} F1={:.4f} '
            'OOV-R={:.4f} IV-R={:.4f}'.format(
                p, r, f1, oov, iv
            )
        )
        

# jieba 作为外部分词基线；环境未安装时跳过。
try:
    import jieba
    jieba_result = [list(jieba.cut(s)) for s in sentences]
    show_metrics('结巴分词', jieba_result)
except ImportError:
    print('未安装 jieba，跳过该基线。')


结巴分词

第 1 句：武汉市长江大桥
标准： ['武汉市', '长江大桥']
预测： ['武汉市', '长江大桥']
P=1.0000 R=1.0000 F1=1.0000 OOV-R=0.0000 IV-R=0.0000

第 2 句：热烈欢迎新老师生前来就餐
标准： ['热烈', '欢迎', '新', '老', '师生', '前来', '就餐']
预测： ['热烈欢迎', '新', '老师', '生前', '来', '就餐']
P=0.3333 R=0.2857 F1=0.3077 OOV-R=0.0000 IV-R=0.0000

第 3 句：近年来自动化研究所取得的成就斐然
标准： ['近年来', '自动化研究所', '取得', '的', '成就斐然']
预测： ['近年来', '自动化', '研究所', '取得', '的', '成就', '斐然']
P=0.4286 R=0.6000 F1=0.5000 OOV-R=0.0000 IV-R=0.0000

第 4 句：门把手坏了
标准： ['门', '把手', '坏', '了']
预测： ['门把手', '坏', '了']
P=0.6667 R=0.5000 F1=0.5714 OOV-R=0.0000 IV-R=0.0000

第 5 句：用正向最大匹配法和逆向最大匹配法进行中文分词的结果是完全一样且正确的
标准： ['用', '正向最大匹配法', '和', '逆向最大匹配法', '进行', '中文', '分词', '的', '结果', '是', '完全', '一样', '且', '正确', '的']
预测： ['用', '正向', '最大', '匹配', '法', '和', '逆向', '最大', '匹配', '法', '进行', '中文', '分词', '的', '结果', '是', '完全', '一样', '且', '正确', '的']
P=0.6190 R=0.8667 F1=0.7222 OOV-R=0.0000 IV-R=0.0000


**请用正向最大匹配算法，结合work文件夹中的workSet词表进行分词，并输出每句话分词结果的精确率、召回率、F1值、OOV召回率、IV召回率**

In [30]:
# 正向最大匹配（FMM）：从左向右寻找最长词。
dictionary = set(word_set)
max_len = max((len(w) for w in dictionary), default=1)

def forward_max_match(sentence, lexicon=dictionary):
    result, i = [], 0
    while i < len(sentence):
        for n in range(min(max_len, len(sentence) - i), 0, -1):
            word = sentence[i:i+n]
            if word in lexicon or n == 1:
                result.append(word)
                i += n
                break
    return result

fmm_result = [forward_max_match(s) for s in sentences]
show_metrics('正向最大匹配（FMM）', fmm_result, dictionary)


正向最大匹配（FMM）

第 1 句：武汉市长江大桥
标准： ['武汉市', '长江大桥']
预测： ['武汉市', '长江', '大桥']
P=0.3333 R=0.5000 F1=0.4000 OOV-R=0.0000 IV-R=1.0000

第 2 句：热烈欢迎新老师生前来就餐
标准： ['热烈', '欢迎', '新', '老', '师生', '前来', '就餐']
预测： ['热烈', '欢迎', '新', '老师', '生前', '来', '就餐']
P=0.5714 R=0.5714 F1=0.5714 OOV-R=0.0000 IV-R=0.5714

第 3 句：近年来自动化研究所取得的成就斐然
标准： ['近年来', '自动化研究所', '取得', '的', '成就斐然']
预测： ['近年来', '自动化', '研究所', '取得', '的', '成就', '斐然']
P=0.4286 R=0.6000 F1=0.5000 OOV-R=0.0000 IV-R=1.0000

第 4 句：门把手坏了
标准： ['门', '把手', '坏', '了']
预测： ['门', '把', '手', '坏', '了']
P=0.6000 R=0.7500 F1=0.6667 OOV-R=0.0000 IV-R=1.0000

第 5 句：用正向最大匹配法和逆向最大匹配法进行中文分词的结果是完全一样且正确的
标准： ['用', '正向最大匹配法', '和', '逆向最大匹配法', '进行', '中文', '分词', '的', '结果', '是', '完全', '一样', '且', '正确', '的']
预测： ['用', '正', '向', '最', '大', '匹', '配', '法', '和', '逆向', '最', '大', '匹', '配', '法', '进行', '中文', '分', '词', '的', '结果', '是', '完全', '一样', '且', '正确', '的']
P=0.4444 R=0.8000 F1=0.5714 OOV-R=0.0000 IV-R=1.0000


**请用逆向最大匹配算法，结合work文件夹中的workSet词表进行分词，并输出每句话分词结果的精确率、召回率、F1值、OOV召回率、IV召回率**

In [31]:
# 逆向最大匹配（BMM）：从右向左寻找最长词。
def backward_max_match(sentence, lexicon=dictionary):
    result, i = [], len(sentence)
    while i > 0:
        for n in range(min(max_len, i), 0, -1):
            word = sentence[i-n:i]
            if word in lexicon or n == 1:
                result.append(word)
                i -= n
                break
    return result[::-1]

bmm_result = [backward_max_match(s) for s in sentences]
# show_metrics('逆向最大匹配（BMM）', bmm_result, dictionary)

**请用双向最大匹配算法，结合work文件夹中的workSet词表进行分词，并输出每句话分词结果的精确率、召回率、F1值、OOV召回率、IV召回率**

In [32]:
# 普通集合词典的双向最大匹配：结合已有的 FMM 和 BMM 结果择优。
def choose_bidirectional(forward, backward):
    # 规则：词数少者优先；词数相同时，单字词少者优先；仍相同则选择正向结果。
    if len(forward) != len(backward):
        return forward if len(forward) < len(backward) else backward

    forward_single_chars = sum(len(word) == 1 for word in forward)
    backward_single_chars = sum(len(word) == 1 for word in backward)
    if forward_single_chars != backward_single_chars:
        return forward if forward_single_chars < backward_single_chars else backward

    return forward

def set_bidirectional_max_match(sentence, lexicon=dictionary):
    forward = forward_max_match(sentence, lexicon)
    backward = backward_max_match(sentence, lexicon)
    return choose_bidirectional(forward, backward)

set_bmm_result = [set_bidirectional_max_match(s) for s in sentences]
# show_metrics('普通词典双向最大匹配', set_bmm_result, dictionary)

In [33]:
# Trie 前缀树实现双向最大匹配：分别执行正向和逆向匹配，再比较结果。
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_word = True

    def longest_match(self, sentence, start):
        """从 start 位置向右查找最长词。"""
        node = self.root
        best_word = None
        for end in range(start, len(sentence)):
            char = sentence[end]
            if char not in node.children:
                break
            node = node.children[char]
            if node.is_word:
                best_word = sentence[start:end + 1]
        return best_word

    def longest_match_backward(self, sentence, end):
        """从 end 位置向左查找最长词。"""
        node = self.root
        best_word = None
        for start in range(end, -1, -1):
            char = sentence[start]
            if char not in node.children:
                break
            node = node.children[char]
            if node.is_word:
                best_word = sentence[start:end + 1]
        return best_word

def trie_forward_max_match(sentence, trie):
    result, i = [], 0
    while i < len(sentence):
        word = trie.longest_match(sentence, i)
        if word is None:
            word = sentence[i]
        result.append(word)
        i += len(word)
    return result

def trie_backward_max_match(sentence, reversed_trie):
    result, end = [], len(sentence) - 1
    while end >= 0:
        reversed_word = reversed_trie.longest_match(
            sentence[::-1], len(sentence) - 1 - end
        )
        if reversed_word is None:
            word = sentence[end]
        else:
            word = reversed_word[::-1]
        result.append(word)
        end -= len(word)
    return result[::-1]

def trie_bidirectional_max_match(sentence, forward_trie, reversed_trie):
    forward = trie_forward_max_match(sentence, forward_trie)
    backward = trie_backward_max_match(sentence, reversed_trie)
    # 词数较少者优先；词数相同时，单字词较少者优先；仍相同则选正向。
    if len(forward) != len(backward):
        return forward if len(forward) < len(backward) else backward
    forward_single_chars = sum(len(word) == 1 for word in forward)
    backward_single_chars = sum(len(word) == 1 for word in backward)
    if forward_single_chars != backward_single_chars:
        return forward if forward_single_chars < backward_single_chars else backward
    return forward

# 创建的两棵树结构示意（以词典含“南京”“南京市”为例）：
# 正向 trie：root -> 南 -> 京* -> 市*（* 表示该节点 is_word=True）
# 反向 reversed_trie：root -> 京 -> 南*；root -> 市 -> 京 -> 南*
# 每个节点的 children 是 {下一字符: 子节点}；不同单词会共享相同前缀路径。
trie = Trie()
reversed_trie = Trie()
for word in dictionary:
    trie.insert(word)             # 原词按从左到右的字符顺序插入正向树
    reversed_trie.insert(word[::-1])  # 反转词按反向查找路径插入反向树

trie_result = [trie_bidirectional_max_match(s, trie, reversed_trie) for s in sentences]
# show_metrics('Trie 双向最大匹配（BMM）', trie_result, dictionary)

**附加任务：1. 重新设计词表结构以加快分词速度，并保存新的词表文件到work文件夹中，对比使用两种词表对于分词执行效率的提升。测试结巴对于专业知识文档的分词能力，通过增加领域词表提升分词效果。**

请点击[此处](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576)查看本环境基本用法.  <br>
Please click [here ](https://ai.baidu.com/docs#/AIStudio_Project_Notebook/a38e5576) for more detailed instructions. 